In [1]:
import json

# Read the collection schemas from JSON file
with open("../data/3-collection-schemas-with-search-property.json", "r") as f:
    collection_schemas = json.load(f)


In [2]:
courses_collection = collection_schemas[2]
print(courses_collection)

{"weaviate_collections":[{"name":"Courses","properties":[{"name":"courseTitle","data_type":["string"],"description":"The title of the course."},{"name":"courseDescription","data_type":["string"],"description":"A detailed summary of the course, including coverage topics and learning outcomes."},{"name":"courseDuration","data_type":["number"],"description":"The total number of hours required to complete the course."},{"name":"currentlyEnrolling","data_type":["boolean"],"description":"Indicates whether the course is currently open for enrollment."}],"envisioned_use_case_overview":"This schema helps users find courses based on subject matter, duration, and enrollment status. Semantic search enhances discovery of courses by learning outcomes and topics covered."},{"name":"Instructors","properties":[{"name":"instructorName","data_type":["string"],"description":"The full name of the instructor."},{"name":"biography","data_type":["string"],"description":"A detailed biography of the instructor,

In [4]:
import os
import weaviate

WEAVIATE_URL = os.getenv("WEAVIATE_URL")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Connecting to Weaviate...")
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY),
    headers={"X-OpenAI-Api-Key": OPENAI_API_KEY},
)
print("Successfully connected to Weaviate...")

# Delete existing collections if they exist
for collection_name in ["Courses", "Instructors", "Students"]:
    if weaviate_client.collections.exists(collection_name):
        weaviate_client.collections.delete(collection_name)

Connecting to Weaviate...
Successfully connected to Weaviate...


In [5]:
import weaviate.classes as wvc

# 1. Create the "Courses" collection
courses_collection = weaviate_client.collections.create(
    name="Courses",
    vectorizer_config=wvc.config.Configure.Vectorizer.text2vec_openai(),
    properties=[
        wvc.config.Property(
            name="courseTitle",
            data_type=wvc.config.DataType.TEXT,
            description="The title of the course."
        ),
        wvc.config.Property(
            name="courseDescription",
            data_type=wvc.config.DataType.TEXT,
            description="A detailed summary of the course, including coverage topics and learning outcomes."
        ),
        wvc.config.Property(
            name="courseDuration",
            data_type=wvc.config.DataType.INT,
            description="The total number of hours required to complete the course."
        ),
        wvc.config.Property(
            name="currentlyEnrolling",
            data_type=wvc.config.DataType.BOOL,
            description="Indicates whether the course is currently open for enrollment."
        ),
    ],
)

# 2. Create the "Instructors" collection
instructors_collection = weaviate_client.collections.create(
    name="Instructors",
    vectorizer_config=wvc.config.Configure.Vectorizer.text2vec_openai(),
    properties=[
        wvc.config.Property(
            name="instructorName",
            data_type=wvc.config.DataType.TEXT,
            description="The full name of the instructor."
        ),
        wvc.config.Property(
            name="biography",
            data_type=wvc.config.DataType.TEXT,
            description="A detailed biography of the instructor, including professional background and teaching philosophy."
        ),
        wvc.config.Property(
            name="yearsOfTeaching",
            data_type=wvc.config.DataType.INT,
            description="The number of years the instructor has been teaching."
        ),
        wvc.config.Property(
            name="tenured",
            data_type=wvc.config.DataType.BOOL,
            description="Indicates whether the instructor holds a tenured position."
        ),
    ],
)

# 3. Create the "Students" collection
students_collection = weaviate_client.collections.create(
    name="Students",
    vectorizer_config=wvc.config.Configure.Vectorizer.text2vec_openai(),
    properties=[
        wvc.config.Property(
            name="studentName",
            data_type=wvc.config.DataType.TEXT,
            description="The full name of the student."
        ),
        wvc.config.Property(
            name="researchInterests",
            data_type=wvc.config.DataType.TEXT,
            description="Detailed information on the student's academic interests and research focus."
        ),
        wvc.config.Property(
            name="completedCredits",
            data_type=wvc.config.DataType.INT,
            description="The number of academic credits the student has completed."
        ),
        wvc.config.Property(
            name="enrolledFullTime",
            data_type=wvc.config.DataType.BOOL,
            description="Indicates whether the student is enrolled full-time."
        ),
    ],
)

print("Created 'Courses', 'Instructors', and 'Students' collections successfully!")

Created 'Courses', 'Instructors', and 'Students' collections successfully!


In [6]:
# Read the CSV files into pandas DataFrames
import pandas as pd

courses_df = pd.read_csv("../data/data-for-use-cases/Courses.csv")
instructors_df = pd.read_csv("../data/data-for-use-cases/Instructors.csv")
students_df = pd.read_csv("../data/data-for-use-cases/Students.csv")

# Convert DataFrames to list of dictionaries with "properties" key
courses_data = [{"properties": row.to_dict()} for _, row in courses_df.iterrows()]
instructors_data = [{"properties": row.to_dict()} for _, row in instructors_df.iterrows()]
students_data = [{"properties": row.to_dict()} for _, row in students_df.iterrows()]

# Insert data into respective collections
for course in courses_data:
    courses_collection.data.insert(properties=course["properties"])

for instructor in instructors_data:
    instructors_collection.data.insert(properties=instructor["properties"])

for student in students_data:
    students_collection.data.insert(properties=student["properties"])

print("Successfully inserted all data into collections!")


Successfully inserted all data into collections!


In [7]:
from pydantic import BaseModel
from typing import Literal, Optional, List

class IntPropertyFilter(BaseModel):
    property_name: str
    operator: Literal["=", "<", ">", "<=", ">="]
    value: int | float


class TextPropertyFilter(BaseModel):
    property_name: str
    operator: Literal["=", "LIKE"]
    value: str


class BooleanPropertyFilter(BaseModel):
    property_name: str
    operator: Literal["=", "!="]
    value: bool


class IntAggregation(BaseModel):
    property_name: str
    metrics: Literal["MIN", "MAX", "MEAN", "MEDIAN", "MODE", "SUM"]


class TextAggregation(BaseModel):
    property_name: str
    metrics: Literal["TOP_OCCURRENCES"]
    top_occurrences_limit: Optional[int] = None


class BooleanAggregation(BaseModel):
    property_name: str
    metrics: Literal["TOTAL_TRUE", "TOTAL_FALSE", "PERCENTAGE_TRUE", "PERCENTAGE_FALSE"]


class WeaviateQuery(BaseModel):
    target_collection: str
    search_query: Optional[str] = None

    integer_property_filter: Optional[IntPropertyFilter] = None
    text_property_filter: Optional[TextPropertyFilter] = None
    boolean_property_filter: Optional[BooleanPropertyFilter] = None

    limit: Optional[int] = 5

    integer_property_aggregation: Optional[IntAggregation] = None
    text_property_aggregation: Optional[TextAggregation] = None
    boolean_property_aggregation: Optional[BooleanAggregation] = None

    groupby_property: Optional[str] = None

In [8]:
from weaviate.collections import Collection
from weaviate.collections.classes.filters import _FilterByProperty
from typing import Any, Union

def _build_filter(
    f: Union[IntPropertyFilter, TextPropertyFilter, BooleanPropertyFilter]
) -> _FilterByProperty:
    """Build a Weaviate filter from a property filter."""
    operators = {
        "=": "equal",
        "!=": "not_equal",
        "<": "less_than",
        ">": "greater_than",
        "<=": "less_or_equal",
        ">=": "greater_or_equal",
        "LIKE": "like",
    }

    method_name = operators.get(f.operator)
    if not method_name:
        raise ValueError(f"Unsupported operator: {f.operator}")

    value = f.value
    if isinstance(f, BooleanPropertyFilter):
        value = bool(value)

    filter_prop = wvc.query.Filter.by_property(f.property_name)
    return getattr(filter_prop, method_name)(value)


def _build_numeric_metric(agg: IntAggregation) -> wvc.query.Metrics:
    """Build numeric metrics for aggregation."""
    metric = wvc.query.Metrics(agg.property_name)

    metric_map = {
        "MIN": lambda m: m.number(minimum=True),
        "MAX": lambda m: m.number(maximum=True),
        "MEAN": lambda m: m.number(mean=True),
        "SUM": lambda m: m.number(sum_=True),
    }

    return metric_map[agg.metrics](metric)


def _build_text_metric(agg: TextAggregation) -> wvc.query.Metrics:
    """Build text metrics for aggregation."""
    metric = wvc.query.Metrics(agg.property_name)

    if agg.metrics == "COUNT":
        return metric.count()
    elif agg.metrics == "TOP_OCCURRENCES":
        return metric.text(
            top_occurrences_count=True,
            top_occurrences_value=True,
        )
    return metric.count()  # default


def _build_boolean_metric(agg: BooleanAggregation) -> wvc.query.Metrics:
    """Build boolean metrics for aggregation."""
    metric = wvc.query.Metrics(agg.property_name)

    metric_map = {
        "TOTAL_TRUE": lambda m: m.boolean(total_true=True),
        "TOTAL_FALSE": lambda m: m.boolean(total_false=True),
        "PERCENTAGE_TRUE": lambda m: m.boolean(percentage_true=True),
        "PERCENTAGE_FALSE": lambda m: m.boolean(percentage_false=True),
    }

    return metric_map[agg.metrics](metric)


def _format_query_result(result: Any) -> str:
    """Format query results into a readable string."""

    # Handle QueryReturn objects (regular search/filter queries)
    if hasattr(result, "objects"):
        formatted = "Found objects:\n"
        for obj in result.objects:
            formatted += "-" * 40 + "\n"
            for key, value in obj.properties.items():
                formatted += f"{key}: {value}\n"
        return formatted

    # Handle AggregateReturn objects (simple aggregations)
    elif hasattr(result, "properties"):
        formatted = "Aggregation results:\n"
        formatted += "-" * 40 + "\n"
        for prop_name, metrics in result.properties.items():
            formatted += f"Property: {prop_name}\n"
            for metric_name, value in metrics.__dict__.items():
                if value is not None:
                    if metric_name == "top_occurrences":
                        formatted += f"  Most common values:\n"
                        for occurrence in value:
                            formatted += f"    - {occurrence.value} (count: {occurrence.count})\n"
                    else:
                        formatted += f"  {metric_name}: {value}\n"
        if hasattr(result, "total_count"):
            formatted += f"Total count: {result.total_count}\n"
        return formatted

    # Handle AggregateGroupByReturn objects (grouped aggregations)
    elif hasattr(result, "groups"):
        formatted = "Grouped aggregation results:\n"
        for group in result.groups:
            formatted += "-" * 40 + "\n"
            formatted += f"Group: {group.grouped_by.prop} = {group.grouped_by.value}\n"
            for prop_name, metrics in group.properties.items():
                formatted += f"Property: {prop_name}\n"
                for metric_name, value in metrics.__dict__.items():
                    if value is not None:
                        if metric_name == "top_occurrences":
                            formatted += f"  Most common values:\n"
                            for occurrence in value:
                                formatted += f"    - {occurrence.value} (count: {occurrence.count})\n"
                        else:
                            formatted += f"  {metric_name}: {value}\n"
            formatted += f"Group count: {group.total_count}\n"
        return formatted

    return str(result)


def execute_weaviate_query(
    collection,
    query: WeaviateQuery,
    return_properties: list[str] | None = None,
) -> str:
    # Build filters if any exist
    filters = None
    if query.integer_property_filter:
        filters = _build_filter(query.integer_property_filter)
    elif query.text_property_filter:
        filters = _build_filter(query.text_property_filter)
    elif query.boolean_property_filter:
        filters = _build_filter(query.boolean_property_filter)

    # Handle aggregations if they exist
    if any([
        query.integer_property_aggregation,
        query.text_property_aggregation,
        query.boolean_property_aggregation,
    ]):
        metrics = []
        if query.integer_property_aggregation:
            metrics.append(_build_numeric_metric(query.integer_property_aggregation))
        if query.text_property_aggregation:
            metrics.append(_build_text_metric(query.text_property_aggregation))
        if query.boolean_property_aggregation:
            metrics.append(_build_boolean_metric(query.boolean_property_aggregation))

        group_by = None
        if query.groupby_property:
            group_by = wvc.aggregate.GroupByAggregate(prop=query.groupby_property)

        if query.search_query:
            result = collection.aggregate.near_text(
                query=query.search_query,
                object_limit=query.limit,
                total_count=True,
                group_by=group_by,
                return_metrics=metrics,
                filters=wvc.query.Filter.all_of([filters]) if filters else None,
            )
        else:
            result = collection.aggregate.over_all(
                total_count=True,
                group_by=group_by,
                return_metrics=metrics,
                filters=wvc.query.Filter.all_of([filters]) if filters else None,
            )
    else:
        # Handle regular queries - use hybrid only when there's a search query
        if query.search_query:
            result = collection.query.hybrid(
                query=query.search_query,
                filters=wvc.query.Filter.all_of([filters]) if filters else None,
                limit=query.limit,
                return_properties=return_properties,
            )
        else:
            # Use fetch() for filter-only queries
            result = collection.query.fetch_objects(
                filters=wvc.query.Filter.all_of([filters]) if filters else None,
                limit=query.limit,
                return_properties=return_properties,
            )

    return _format_query_result(result)


def query_collection(weaviate_client, query: WeaviateQuery) -> str:
    """Query Weaviate, Return Search Results or Aggregations."""
    collection = weaviate_client.collections.get(query.target_collection)
    return execute_weaviate_query(
        collection=collection,
        query=query,
        return_properties=None
    )

In [10]:
# Test case 1: Simple search query
simple_search = WeaviateQuery(
    target_collection="Courses", 
    search_query="machine learning",
    limit=5
)

# Test case 2: Query with integer filter
numeric_filter_query = WeaviateQuery(
    target_collection="Courses",
    integer_property_filter=IntPropertyFilter(
        property_name="courseDuration",
        operator=">",
        value=40
    ),
    limit=3
)

# Test case 3: Query with text filter and aggregation
text_filter_and_agg_query = WeaviateQuery(
    target_collection="Students",
    boolean_property_filter=BooleanPropertyFilter(
        property_name="enrolledFullTime",
        operator="=",
        value=True
    ),
    text_property_aggregation=TextAggregation(
        property_name="researchInterests",
        metrics="TOP_OCCURRENCES",
        top_occurrences_limit=5
    )
)

# Test case 4: Query with boolean filter and aggregation with groupby
complex_query = WeaviateQuery(
    target_collection="Instructors",
    boolean_property_filter=BooleanPropertyFilter(
        property_name="tenured",
        operator="=",
        value=True
    ),
    integer_property_aggregation=IntAggregation(
        property_name="yearsOfTeaching",
        metrics="MEAN"
    ),
    groupby_property="tenured"
)

# Function to run all test cases
def run_test_cases(weaviate_client):
    test_cases = [
        ("Simple Search", simple_search),
        ("Numeric Filter", numeric_filter_query),
        ("Text Filter with Aggregation", text_filter_and_agg_query),
        ("Complex Query", complex_query)
    ]
    
    for test_name, query in test_cases:
        print(f"\nTesting: {test_name}")
        print("-" * 50)
        try:
            result = query_collection(weaviate_client, query)
            print("Success!")
            print("Result:")
            print(result)
        except Exception as e:
            print(f"Error occurred: {str(e)}")
            print(f"Query details: {query.dict()}")

run_test_cases(weaviate_client)


Testing: Simple Search
--------------------------------------------------
Success!
Result:
Found objects:
----------------------------------------
courseDescription: Deep dive into neural networks, reinforcement learning, and deep learning architectures. Includes hands-on projects with real-world datasets and implementation of state-of-the-art algorithms. Focus on both theoretical foundations and practical applications.
courseDuration: 48
currentlyEnrolling: True
courseTitle: Advanced Machine Learning
----------------------------------------
courseDescription: In-depth study of Mathematics Linear Algebra. includes hands-on projects and features case studies. Prepares students for professional practice.
courseDuration: 34
currentlyEnrolling: True
courseTitle: Linear Algebra II
----------------------------------------
courseDescription: Interactive learning experience focusing on Computer Science Data Structures. combines theoretical and practical elements and incorporates real-world ap

In [11]:
import dspy

lm = dspy.LM(model="openai/gpt-4o", api_key=os.getenv("OPENAI_API_KEY"))
dspy.settings.configure(lm=lm)
lm("say hello")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)


['Hello! How can I assist you today?']

In [21]:
class SearchOrRespond(dspy.Signature):
    """Either perform a Weaviate search or respond directly based on the query."""
    
    user_query: str = dspy.InputField()
    
    database_schema: str = dspy.InputField(desc="The database schema you can query to gain necessary information to provide the most accurate possible to the user.")
    
    database_querying_manual: str = dspy.InputField(desc="A collection of notes about how to query the database and what kind of information you can access from its operators and the data contained in the collection described by the `database_schema` input.")

    query_execution_history: str = dspy.InputField(desc="History of previous queries and their responses to provide context for the current query")

    should_search: bool = dspy.OutputField(desc="True if we need to search Weaviate, False if we can answer directly")
    search_parameters: Optional[WeaviateQuery] = dspy.OutputField(desc="Parameters for WeaviateQuery if should_search is True")
    response_to_user: Optional[str] = dspy.OutputField(desc="A succinct and information dense response to the User.")

class UpdateQueryingManual(dspy.Signature):
    """Determine if and how to update the querying manual based on recent experience. 
    IMPORTANT: The manual MUST NOT exceed 16 sentences total!"""
    
    current_manual: str = dspy.InputField()
    query: str = dspy.InputField()
    result: str = dspy.InputField()
    updated_manual: str = dspy.OutputField(desc="The updated manual incorporating new insights. CRITICAL: Must be 16 sentences or fewer!")

class TrimQueryingManual(dspy.Signature):
    """Trim and organize the querying manual to ensure it stays within length limits."""
    
    current_manual: str = dspy.InputField()
    trimmed_manual: str = dspy.OutputField(desc="A concise, well-organized version of the manual containing only the most essential information in 16 sentences or fewer.")

class FormatResponse(dspy.Signature):
    """Format the raw response into a natural, conversational reply."""
    
    raw_response: str = dspy.InputField()
    user_query: str = dspy.InputField()
    formatted_response: str = dspy.OutputField(desc="A naturally worded, friendly response that presents the information clearly and engagingly.")

class WeaviateFunctionCallingAgent(dspy.Module):
    def __init__(self, weaviate_client, collections_description):
        self.predict = dspy.Predict(SearchOrRespond)
        self.update_manual = dspy.Predict(UpdateQueryingManual)
        self.trim_manual = dspy.Predict(TrimQueryingManual)
        self.format_response = dspy.Predict(FormatResponse)
        self.weaviate_client = weaviate_client
        self.collections_description = collections_description
        self.query_history = ""
        self.database_querying_manual = "Initial manual for querying the database."
    
    def get_database_querying_manual(self) -> str:
        """Inspect learned database manual."""
        return self.database_querying_manual

    def forward(self, user_query: str) -> str:
        # Determine whether to search or respond directly
        prediction = self.predict(
            user_query=user_query, 
            database_schema=self.collections_description,
            database_querying_manual=self.database_querying_manual,
            query_execution_history=self.query_history
        )
        
        result = None
        if prediction.should_search:
            query = WeaviateQuery(
                target_collection=prediction.search_parameters.target_collection,
                search_query=prediction.search_parameters.search_query,
                integer_property_filter=prediction.search_parameters.integer_property_filter,
                text_property_filter=prediction.search_parameters.text_property_filter,
                boolean_property_filter=prediction.search_parameters.boolean_property_filter,
                limit=prediction.search_parameters.limit,
                integer_property_aggregation=prediction.search_parameters.integer_property_aggregation,
                text_property_aggregation=prediction.search_parameters.text_property_aggregation,
                boolean_property_aggregation=prediction.search_parameters.boolean_property_aggregation,
                groupby_property=prediction.search_parameters.groupby_property
            )
            result = query_collection(self.weaviate_client, query)
            # Update query history
            self.query_history += f"\nQuery: {user_query}\nResult: {result}\n"
            
            # Update manual with default if none returned
            manual_update = self.update_manual(
                current_manual=self.database_querying_manual,
                query=user_query,
                result=str(result)
            )
            if manual_update and manual_update.updated_manual:
                self.database_querying_manual = manual_update.updated_manual
                # Trim manual if needed
                trimmed = self.trim_manual(current_manual=self.database_querying_manual)
                if trimmed and trimmed.trimmed_manual:
                    self.database_querying_manual = trimmed.trimmed_manual
            
            # Format the response
            formatted = self.format_response(raw_response=str(result), user_query=user_query)
            return formatted.formatted_response if formatted else result
        else:
            # Update manual with default if none returned
            manual_update = self.update_manual(
                current_manual=self.database_querying_manual,
                query=user_query,
                result=prediction.response_to_user
            )
            if manual_update and manual_update.updated_manual:
                self.database_querying_manual = manual_update.updated_manual
                # Trim manual if needed
                trimmed = self.trim_manual(current_manual=self.database_querying_manual)
                if trimmed and trimmed.trimmed_manual:
                    self.database_querying_manual = trimmed.trimmed_manual
            
            # Format the direct response
            formatted = self.format_response(raw_response=prediction.response_to_user, user_query=user_query)
            return dspy.Prediction(response=formatted.formatted_response if formatted else prediction.response_to_user)

In [22]:
schema = '''{"weaviate_collections":[{"name":"Courses","properties":[{"name":"courseTitle","data_type":["string"],"description":"The title of the course."},{"name":"courseDescription","data_type":["string"],"description":"A detailed summary of the course, including coverage topics and learning outcomes."},{"name":"courseDuration","data_type":["number"],"description":"The total number of hours required to complete the course."},{"name":"currentlyEnrolling","data_type":["boolean"],"description":"Indicates whether the course is currently open for enrollment."}],"envisioned_use_case_overview":"This schema helps users find courses based on subject matter, duration, and enrollment status. Semantic search enhances discovery of courses by learning outcomes and topics covered."},{"name":"Instructors","properties":[{"name":"instructorName","data_type":["string"],"description":"The full name of the instructor."},{"name":"biography","data_type":["string"],"description":"A detailed biography of the instructor, including professional background and teaching philosophy."},{"name":"yearsOfTeaching","data_type":["number"],"description":"The number of years the instructor has been teaching."},{"name":"tenured","data_type":["boolean"],"description":"Indicates whether the instructor holds a tenured position."}],"envisioned_use_case_overview":"This schema allows students and administrators to search for instructors based on experience and background. Rich biographies help in matching students with instructors who align with their learning style and academic goals."},{"name":"Students","properties":[{"name":"studentName","data_type":["string"],"description":"The full name of the student."},{"name":"researchInterests","data_type":["string"],"description":"Detailed information on the student's academic interests and research focus."},{"name":"completedCredits","data_type":["number"],"description":"The number of academic credits the student has completed."},{"name":"enrolledFullTime","data_type":["boolean"],"description":"Indicates whether the student is enrolled full-time."}],"envisioned_use_case_overview":"This schema is designed to help institutions manage student data and preferences. Semantic search allows deeper insights into student research interests and progression paths."}]}'''

weaviate_search_module = WeaviateFunctionCallingAgent(
    weaviate_client=weaviate_client,
    collections_description=schema
)

print(weaviate_search_module.get_database_querying_manual())

Initial manual for querying the database.


In [23]:
import csv
import os
from datetime import datetime

# Define questions list
questions = [
    # About Courses
    "Which courses are currently open for enrollment?",
    "Show me all courses where the total duration is less than 10 hours.",
    "Which courses mention advanced Python programming in their description?", 
    "What are the key learning outcomes for the course titled 'Data Structures 101'?",
    "List all courses that have a duration of 40 hours or more.",

    # About Instructors
    "Which instructors have over 10 years of teaching experience?",
    "Show me the biography of the instructor named 'Dr. Jane Doe'.",
    "Which instructors are tenured?",
    "Who has the longest teaching career among all instructors in the database?",
    "Find instructors who mention a 'hands-on learning' philosophy in their biography.",

    # About Students
    "List all students who are enrolled full-time.",
    "Which students have completed more than 30 credits?",
    "Show me the research interests of the student named 'Alex Johnson'.",
    "Find students with research interests in machine learning or data science.",
    "Which students are pursuing studies in artificial intelligence?",

    # Combining Criteria
    "Which instructors have a biography mentioning 'online teaching methods' and have been teaching for more than 5 years?",
    "Which courses currently open for enrollment have a duration under 20 hours?",
    "Show me all students who have completed at least 20 credits but are not enrolled full-time.",
    "Which instructors are tenured and have a teaching philosophy related to 'project-based learning'?",
    "Which courses specifically mention 'capstone project' in their description and are currently enrolling?"
]

csv_filename = "meta-learning-results.csv"

print(f"Starting processing of {len(questions)} questions...")
weaviate_search_module = WeaviateFunctionCallingAgent(
    weaviate_client=weaviate_client,
    collections_description=schema
)

with open(csv_filename, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Question Number', 'Question', 'Manual State'])
    
    # Process each question and track manual evolution
    for i, question in enumerate(questions, 1):
        print(f"\nProcessing question {i}/{len(questions)}: {question}")
        try:
            prediction = weaviate_search_module(question)
            manual_state = weaviate_search_module.get_database_querying_manual()
            writer.writerow([i, question, manual_state])
            print(f"✓ Question {i} processed successfully")
            print(f"  Response length: {len(str(prediction))} characters")
            print(f"  Manual length: {len(manual_state)} characters")
        except Exception as e:
            print(f"✗ Error processing question {i}: {str(e)}")

print(f"\nProcessing complete. Results saved to {csv_filename}")


Starting processing of 20 questions...

Processing question 1/20: Which courses are currently open for enrollment?
✓ Question 1 processed successfully
  Response length: 1547 characters
  Manual length: 600 characters

Processing question 2/20: Show me all courses where the total duration is less than 10 hours.
✓ Question 2 processed successfully
  Response length: 171 characters
  Manual length: 709 characters

Processing question 3/20: Which courses mention advanced Python programming in their description?
✓ Question 3 processed successfully
  Response length: 250 characters
  Manual length: 808 characters

Processing question 4/20: What are the key learning outcomes for the course titled 'Data Structures 101'?
✓ Question 4 processed successfully
  Response length: 222 characters
  Manual length: 811 characters

Processing question 5/20: List all courses that have a duration of 40 hours or more.
✓ Question 5 processed successfully
  Response length: 2876 characters
  Manual length: 9

In [24]:
print(weaviate_search_module.get_database_querying_manual())

To find open courses, query "Which courses are currently open for enrollment?" and ensure 'currentlyEnrolling' is 'True'. Use duration filters for courses under 10, 20, or 40 hours. Include keywords in descriptions to search by topic, such as "capstone project". Verify course titles for key outcomes if no results appear. For experienced instructors, query "Which instructors have over 10 years of teaching experience?" and check 'yearsOfTeaching'. View an instructor's biography by querying "Show me the biography of the instructor named [Instructor's Name]" with correct spelling. Identify tenured instructors by querying "Which instructors are tenured?" and ensure 'tenured' is 'True'. To find the instructor with the longest career, use "Who has the longest teaching career among all instructors?" and check 'yearsOfTeaching' for the maximum value. For instructors with a 'hands-on learning' philosophy, query "Which instructors mention a 'hands-on learning' philosophy in their biography?" and 

In [25]:
weaviate_search_module("How many students are enrolled in machine learning courses?")

'We have two students currently involved in machine learning-related research. Christopher Perez is focusing on using machine learning to detect financial fraud and optimize risk assessment, while Aubrey Bennett is working on developing algorithms for personalized dietary recommendations. Christopher is enrolled part-time, having completed 72 credits, and Aubrey is enrolled full-time with 33 credits completed.'